# Lunar M³ Project — Exploration

This notebook demonstrates the end-to-end synthetic pipeline and basic plots.


In [ ]:
import numpy as np
import pandas as pd

from lunar_m3.data_loading import generate_synthetic_cube
from lunar_m3.features import extract_feature_table
from lunar_m3.models import train_baseline_classifier
from lunar_m3.visualization import plot_label_map, plot_spectrum_comparison
from lunar_m3.preprocessing import clip_invalid_reflectance, normalize_by_reference_window, savgol_smooth


In [ ]:
cube = generate_synthetic_cube(rows=60, cols=80, bands=86, seed=0)
features = extract_feature_table(cube.data, cube.wavelengths)
y = np.asarray(getattr(cube, 'synthetic_labels')).reshape(-1)
result = train_baseline_classifier(features, y, model='logreg', random_state=0)
print(result.report)


In [ ]:
rows, cols, _ = cube.data.shape
pred = result.pipeline.predict(features[[c for c in features.columns if c not in {'x','y'}]])
plot_label_map(pred.reshape(rows, cols), title='Predicted classes (synthetic)')


In [ ]:
x0, y0 = cols // 2, rows // 2
raw = cube.get_pixel_spectrum(x0, y0)
proc = clip_invalid_reflectance(raw)
proc = normalize_by_reference_window(cube.wavelengths, proc)
proc = savgol_smooth(proc)
plot_spectrum_comparison(cube.wavelengths, raw, proc, title=f'Pixel spectrum (x={x0}, y={y0})')
